<a href="https://colab.research.google.com/github/Shubhiii16/Meal-app-public/blob/main/Capstone_Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import hashlib
import json
import time
from datetime import datetime
from typing import Dict, List, Optional
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend


class Block:
    """Represents a block in the blockchain"""

    def __init__(self, index: int, timestamp: float, data: Dict, previous_hash: str):
        self.index = index
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = 0
        self.hash = self.calculate_hash()

    def calculate_hash(self) -> str:
        """Calculate SHA-256 hash of the block"""
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

    def mine_block(self, difficulty: int):
        """Mine block with proof of work"""
        target = "0" * difficulty
        while self.hash[:difficulty] != target:
            self.nonce += 1
            self.hash = self.calculate_hash()


class DecentralizedIdentity:
    """Represents a decentralized identity (DID)"""

    def __init__(self, name: str, email: str):
        self.did = self._generate_did()
        self.name = name
        self.email = email
        self.created_at = datetime.now().isoformat()
        self.private_key, self.public_key = self._generate_keys()
        self.credentials = []
        self.verified = False

    def _generate_did(self) -> str:
        """Generate unique DID"""
        unique_string = f"{time.time()}{hashlib.sha256(str(time.time()).encode()).hexdigest()}"
        did_hash = hashlib.sha256(unique_string.encode()).hexdigest()[:32]
        return f"did:blockchain:{did_hash}"

    def _generate_keys(self):
        """Generate RSA key pair"""
        private_key = rsa.generate_private_key(
            public_exponent=65537,
            key_size=2048,
            backend=default_backend()
        )
        public_key = private_key.public_key()
        return private_key, public_key

    def get_public_key_pem(self) -> str:
        """Export public key as PEM string"""
        return self.public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ).decode()

    def sign_data(self, data: str) -> bytes:
        """Sign data with private key"""
        return self.private_key.sign(
            data.encode(),
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH
            ),
            hashes.SHA256()
        )

    def to_dict(self) -> Dict:
        """Convert identity to dictionary"""
        return {
            "did": self.did,
            "name": self.name,
            "email": self.email,
            "created_at": self.created_at,
            "public_key": self.get_public_key_pem(),
            "credentials": self.credentials,
            "verified": self.verified
        }


class Credential:
    """Represents a verifiable credential"""

    def __init__(self, issuer_did: str, subject_did: str, credential_type: str, claims: Dict):
        self.id = self._generate_credential_id()
        self.issuer_did = issuer_did
        self.subject_did = subject_did
        self.credential_type = credential_type
        self.claims = claims
        self.issued_at = datetime.now().isoformat()
        self.signature = None

    def _generate_credential_id(self) -> str:
        """Generate unique credential ID"""
        unique_string = f"{time.time()}{hashlib.sha256(str(time.time()).encode()).hexdigest()}"
        return f"cred:{hashlib.sha256(unique_string.encode()).hexdigest()[:24]}"

    def to_dict(self) -> Dict:
        """Convert credential to dictionary"""
        return {
            "id": self.id,
            "issuer_did": self.issuer_did,
            "subject_did": self.subject_did,
            "type": self.credential_type,
            "claims": self.claims,
            "issued_at": self.issued_at
        }


class IdentityBlockchain:
    """Blockchain for managing decentralized identities"""

    def __init__(self, difficulty: int = 2):
        self.chain: List[Block] = []
        self.difficulty = difficulty
        self.identities: Dict[str, DecentralizedIdentity] = {}
        self.pending_transactions = []
        self._create_genesis_block()

    def _create_genesis_block(self):
        """Create the first block in the chain"""
        genesis_block = Block(0, time.time(), {"type": "genesis"}, "0")
        genesis_block.mine_block(self.difficulty)
        self.chain.append(genesis_block)

    def get_latest_block(self) -> Block:
        """Get the most recent block"""
        return self.chain[-1]

    def create_identity(self, name: str, email: str) -> DecentralizedIdentity:
        """Create a new decentralized identity"""
        identity = DecentralizedIdentity(name, email)
        self.identities[identity.did] = identity

        # Add identity creation to blockchain
        transaction_data = {
            "type": "identity_creation",
            "did": identity.did,
            "name": name,
            "email": email,
            "public_key": identity.get_public_key_pem(),
            "timestamp": identity.created_at
        }
        self.add_transaction(transaction_data)

        return identity

    def issue_credential(self, issuer_did: str, subject_did: str,
                        credential_type: str, claims: Dict) -> Optional[Credential]:
        """Issue a verifiable credential"""
        if issuer_did not in self.identities or subject_did not in self.identities:
            print("Error: Issuer or subject DID not found")
            return None

        issuer = self.identities[issuer_did]
        credential = Credential(issuer_did, subject_did, credential_type, claims)

        # Sign the credential
        credential_data = json.dumps(credential.to_dict(), sort_keys=True)
        credential.signature = issuer.sign_data(credential_data).hex()

        # Add credential to subject's identity
        self.identities[subject_did].credentials.append(credential.to_dict())

        # Add to blockchain
        transaction_data = {
            "type": "credential_issuance",
            "credential": credential.to_dict(),
            "signature": credential.signature
        }
        self.add_transaction(transaction_data)

        return credential

    def verify_identity(self, did: str, verifier_did: str) -> bool:
        """Verify an identity"""
        if did not in self.identities or verifier_did not in self.identities:
            return False

        self.identities[did].verified = True

        transaction_data = {
            "type": "identity_verification",
            "verified_did": did,
            "verifier_did": verifier_did,
            "timestamp": datetime.now().isoformat()
        }
        self.add_transaction(transaction_data)

        return True

    def add_transaction(self, transaction_data: Dict):
        """Add a transaction to pending transactions"""
        self.pending_transactions.append(transaction_data)

    def mine_pending_transactions(self):
        """Mine all pending transactions into a new block"""
        if not self.pending_transactions:
            print("No transactions to mine")
            return

        block = Block(
            len(self.chain),
            time.time(),
            {"transactions": self.pending_transactions},
            self.get_latest_block().hash
        )
        block.mine_block(self.difficulty)
        self.chain.append(block)

        print(f"Block #{block.index} mined: {block.hash}")
        self.pending_transactions = []

    def get_identity(self, did: str) -> Optional[DecentralizedIdentity]:
        """Retrieve an identity by DID"""
        return self.identities.get(did)

    def validate_chain(self) -> bool:
        """Validate the entire blockchain"""
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i - 1]

            if current_block.hash != current_block.calculate_hash():
                return False

            if current_block.previous_hash != previous_block.hash:
                return False

        return True

    def display_chain(self):
        """Display the blockchain"""
        print("\n" + "="*70)
        print("BLOCKCHAIN - DECENTRALIZED IDENTITY MANAGEMENT")
        print("="*70)

        for block in self.chain:
            print(f"\nBlock #{block.index}")
            print(f"Timestamp: {datetime.fromtimestamp(block.timestamp)}")
            print(f"Previous Hash: {block.previous_hash}")
            print(f"Hash: {block.hash}")
            print(f"Nonce: {block.nonce}")
            print(f"Data: {json.dumps(block.data, indent=2)}")
            print("-" * 70)

    def display_identities(self):
        """Display all registered identities"""
        print("\n" + "="*70)
        print("REGISTERED IDENTITIES")
        print("="*70)

        for did, identity in self.identities.items():
            print(f"\nDID: {did}")
            print(f"Name: {identity.name}")
            print(f"Email: {identity.email}")
            print(f"Verified: {identity.verified}")
            print(f"Credentials: {len(identity.credentials)}")
            print("-" * 70)


# Demo Usage
if __name__ == "__main__":
    print("Initializing Decentralized Identity Management System...")
    blockchain = IdentityBlockchain(difficulty=2)

    # Create identities
    print("\n1. Creating identities...")
    alice = blockchain.create_identity("Alice Johnson", "alice@example.com")
    bob = blockchain.create_identity("Bob Smith", "bob@example.com")
    university = blockchain.create_identity("MIT University", "admin@mit.edu")

    print(f"✓ Alice's DID: {alice.did}")
    print(f"✓ Bob's DID: {bob.did}")
    print(f"✓ University's DID: {university.did}")

    # Mine the identity creation transactions
    print("\n2. Mining identity creation transactions...")
    blockchain.mine_pending_transactions()

    # Issue credentials
    print("\n3. Issuing credentials...")
    degree_credential = blockchain.issue_credential(
        university.did,
        alice.did,
        "EducationCredential",
        {
            "degree": "Bachelor of Science",
            "major": "Computer Science",
            "graduation_year": "2024",
            "gpa": "3.85"
        }
    )
    print(f"✓ Degree credential issued to Alice")

    # Verify identity
    print("\n4. Verifying identity...")
    blockchain.verify_identity(alice.did, university.did)
    print(f"✓ Alice's identity verified by University")

    # Mine pending transactions
    print("\n5. Mining credential and verification transactions...")
    blockchain.mine_pending_transactions()

    # Display results
    blockchain.display_identities()
    blockchain.display_chain()

    # Validate blockchain
    print("\n6. Validating blockchain integrity...")
    is_valid = blockchain.validate_chain()
    print(f"Blockchain valid: {is_valid}")

    print("\n" + "="*70)
    print("Demo completed successfully!")
    print("="*70)

Initializing Decentralized Identity Management System...

1. Creating identities...
✓ Alice's DID: did:blockchain:c8865bf9066f561dcdf37179d6101a2d
✓ Bob's DID: did:blockchain:c7766f3a4d7161a6f06db45ed5f2d5ad
✓ University's DID: did:blockchain:56c6daf6c006d8c52d58b15016f9489a

2. Mining identity creation transactions...
Block #1 mined: 009fa9c5bc7a15113671f97f6054556620dd512a2597ad3187fd2ff0b89b1bee

3. Issuing credentials...
✓ Degree credential issued to Alice

4. Verifying identity...
✓ Alice's identity verified by University

5. Mining credential and verification transactions...
Block #2 mined: 00257f3ab9851f75cd8cf2374f2c31a399262fbd1f3726c4248923994473570d

REGISTERED IDENTITIES

DID: did:blockchain:c8865bf9066f561dcdf37179d6101a2d
Name: Alice Johnson
Email: alice@example.com
Verified: True
Credentials: 1
----------------------------------------------------------------------

DID: did:blockchain:c7766f3a4d7161a6f06db45ed5f2d5ad
Name: Bob Smith
Email: bob@example.com
Verified: False